# Exercise 9 - Tax planning

In the previous exercise, you created a data frame representing your store’s products and sales. In this exercise, you will extend that data frame (literally). It’s pretty com mon to add columns to an existing data frame, either to add new information you’ve acquired or to store the results of per-row calculations—which is what you’ll do now. A common reason to add a column is to hold intermediate values as a convenience.

The backstory for this exercise is as follows. Your local government is thinking about imposing a sales tax and is considering 15%, 20%, and 25% rates. Show how much less you would net with each of these tax amounts by adding columns to the data frame for your net income under each of the proposed rates, as well as your current net income.

1. Create DataFrame from exercise 8

In [1]:
from pandas import DataFrame

inventory = DataFrame(
    [
      [11, "keyboard", 10.00, 15.00, 20],
      [55, "Mouse", 5.00, 10.00, 32],
      [30, "Bluetooth Speaker", 20.00, 35.00, 50],
      [25, "Tablet", 200.00, 450.00, 10],
      [13, "Laptop", 500.00, 800.00, 36],
     ],
    columns=["product_id", "product_name", "wholesale_price", "retail_price", "sales"]
)

print(inventory)

   product_id       product_name  wholesale_price  retail_price  sales
0          11           keyboard             10.0          15.0     20
1          55              Mouse              5.0          10.0     32
2          30  Bluetooth Speaker             20.0          35.0     50
3          25             Tablet            200.0         450.0     10
4          13             Laptop            500.0         800.0     36


2. Calculate revenue and assign it to a column of ```inventory``` DataFrame

In [31]:
inventory["revenue"] = ((inventory["retail_price"] - inventory['wholesale_price']) * inventory["sales"])

print(inventory)

   product_id       product_name  wholesale_price  retail_price  sales  \
0          11           keyboard            10.00         15.00     20   
1          55              Mouse             5.00         10.00     32   
2          30  Bluetooth Speaker            20.00         35.00     50   
3          25             Tablet           200.00        450.00     10   
4          13             Laptop           500.00        800.00     36   

    revenue  
0    100.00  
1    160.00  
2    750.00  
3  2,500.00  
4 10,800.00  


3. Calculate taxes of 15%, 20% and 25% and assign them to columns of ```inventory``` DataFrame

In [32]:
inventory["after_15"] = inventory["revenue"] * 0.85 # 15% tax
inventory["after_20"] = inventory["revenue"] * 0.8 # 20% tax
inventory["after_25"] = inventory["revenue"] * 0.75 # 25% tax

print(inventory)

   product_id       product_name  wholesale_price  retail_price  sales  \
0          11           keyboard            10.00         15.00     20   
1          55              Mouse             5.00         10.00     32   
2          30  Bluetooth Speaker            20.00         35.00     50   
3          25             Tablet           200.00        450.00     10   
4          13             Laptop           500.00        800.00     36   

    revenue  after_15  after_20  after_25  
0    100.00     85.00     80.00     75.00  
1    160.00    136.00    128.00    120.00  
2    750.00    637.50    600.00    562.50  
3  2,500.00  2,125.00  2,000.00  1,875.00  
4 10,800.00  9,180.00  8,640.00  8,100.00  


4. Calculate how much we would earn under each tax plan.

In [33]:
earnings = inventory[["revenue", "after_15", "after_20", "after_25"]].sum() # sum works over each column
taxes = inventory["revenue"].sum() - inventory[["after_15", "after_20", "after_25"]].sum()

print(f"Earnings under each tax plan:\n{earnings}\n\nTaxes under each tax plan:\n{taxes}\n")

Earnings under each tax plan:
revenue    14,310.00
after_15   12,163.50
after_20   11,448.00
after_25   10,732.50
dtype: float64

Taxes under each tax plan:
after_15   2,146.50
after_20   2,862.00
after_25   3,577.50
dtype: float64



## Beyond the exercise

* An alternative tax plan would charge a 25% tax, but only on products from which you would net more than 10,000. In such a case, how much would you make?

In [34]:
inventory.drop(columns=["after_15", "after_20", "after_25"], inplace=True) # remove taxes columns

In [35]:
# calculate 25% tax on sales over $10000.0
inventory["after_25"] = inventory["revenue"].mask(inventory["revenue"] > 10000.0, inventory["revenue"] * 0.75)

print(f"25% tax on sales over $10000.0\n{inventory}\n\nTotal earnings: ${inventory['after_25'].sum()}")


25% tax on sales over $10000.0
   product_id       product_name  wholesale_price  retail_price  sales  \
0          11           keyboard            10.00         15.00     20   
1          55              Mouse             5.00         10.00     32   
2          30  Bluetooth Speaker            20.00         35.00     50   
3          25             Tablet           200.00        450.00     10   
4          13             Laptop           500.00        800.00     36   

    revenue  after_25  
0    100.00    100.00  
1    160.00    160.00  
2    750.00    750.00  
3  2,500.00  2,500.00  
4 10,800.00  8,100.00  

Total earnings: $11610.0


* Yet another alternative tax plan would charge a 25% tax on products whose retail price is greater than 80, a 10% tax on products whose retail price is between 30 and 80, and no tax on other products. Implement and calculate the result of such a tax scheme.

In [36]:
inventory.drop(columns=["after_25", ], inplace=True) # remove tax column

In [37]:
inventory["after_taxes"] = inventory["revenue"].mask( # 25% tax on products with retail price > $80.0
    inventory["retail_price"] > 80.0, inventory['revenue'] * 0.75
).mask( # 10% tax on products with retail price between $30.0 and $80.0 (inclusive)
    (inventory["retail_price"] > 30.0) & (inventory["retail_price"] <= 80.0), inventory["revenue"] * 0.90
)

print(f"25% tax for retail prices > $80 and 10% tax for retail prices in $(30.0, 80.0] range:\n{inventory}\n\n"
      f"Total earnings: ${inventory['after_taxes'].sum()}")


25% tax for retail prices > $80 and 10% tax for retail prices in $(30.0, 80.0] range:
   product_id       product_name  wholesale_price  retail_price  sales  \
0          11           keyboard            10.00         15.00     20   
1          55              Mouse             5.00         10.00     32   
2          30  Bluetooth Speaker            20.00         35.00     50   
3          25             Tablet           200.00        450.00     10   
4          13             Laptop           500.00        800.00     36   

    revenue  after_taxes  
0    100.00       100.00  
1    160.00       160.00  
2    750.00       675.00  
3  2,500.00     1,875.00  
4 10,800.00     8,100.00  

Total earnings: $10910.0


* These long floating-point numbers are getting hard to read. Set the ```float_format``` option in pandas such that floating-point numbers will be displayed with commas every three digits before the decimal point and only two digits after the decimal point. Note that this is tricky because it requires understanding Python callables and the ```str.format``` method.

In [38]:
import pandas

pandas.options.display.float_format = "{:,.2f}".format # use commas for thousands separator, and 2 decimal places

In [39]:
inventory # show inventory with the new formating

,product_id,product_name,wholesale_price,retail_price,sales,revenue,after_taxes
0,11,keyboard,10.00,15.00,20,100.00,100.00
1,55,Mouse,5.00,10.00,32,160.00,160.00
2,30,Bluetooth Speaker,20.00,35.00,50,750.00,675.00
3,25,Tablet,200.00,450.00,10,"2,500.00","1,875.00"
4,13,Laptop,500.00,800.00,36,"10,800.00","8,100.00"
